In [1]:
import torch
from torch import float64
from torch.functional import F
import torchvision

In [10]:
class NeuralNetwork():
    def __init__(self, input_features, hidden_layer1, hidden_layer2, num_classes, bias_value, learning_rate) -> None:
        self.num_classes = num_classes
        self.learning_rate = learning_rate

        self.W1 = torch.randn((input_features, hidden_layer1), dtype=torch.float32)
        self.W2 = torch.randn((hidden_layer1, hidden_layer2), dtype=torch.float32)
        self.W3 = torch.randn((hidden_layer2, num_classes), dtype=torch.float32)

        self.b1 = torch.full((hidden_layer1,), bias_value, dtype=torch.float32)
        self.b2 = torch.full((hidden_layer2,), bias_value, dtype=torch.float32)
        self.b3 = torch.full((num_classes,), bias_value, dtype=torch.float32)
    
    def tanh(self, t):
        return torch.tanh(t)
    
    def tanh_grad(self, t):
        return 1 - t**2

    def relu(self, r):
        return torch.relu()
    
    def relu_grad(self, r):
        return torch.where(r > 0, 1, 0)
    
    def softmax(self, s):
        return torch.softmax(s)

    def loss(self, pred, y):
        N = y.shape[0]
        one_hot_y = F.one_hot(y, self.num_classes).float()

        return -torch.sum(one_hot_y * torch.log(pred)) / N

    def forward(self, X):
        
        H1 = X @ self.W1 + self.b1 # (N, input_features) @ (input_feature, hidden_layer1) = (N, hidden_layer1)
        A1 = self.tanh(H1) # (N, hidden_layer1)

        H2 = A1 @ self.W2 + self.b2 # (N, hidden_layer1) @ (hidden_layer1, hidden_layer2) = (N, hidden_layer2)
        A2 = self.relu(H2) # (N, hidden_layer2)

        H3 = A2 @ self.W3 + self.b3 # (N, hidden_layer2) @ (hidden_layer2, num_classes) = (N, num_classes)

        predictions = self.softmax(H3) # (N, num_classes)

        cache = (H1, A1, H2, A2, H3)
        return predictions, cache
    
    def simple_parameter_update(self, parameter, grad, learning_rate):
        new_parameter -= learning_rate * grad
        return new_parameter
    
    def update_parameters(self, parameter, grad, learning_rate, config = None, strategy = 'simple'):
        if strategy == 'simple':
            return self.simple_parameter_update(parameter, grad, learning_rate)
    
    def backward(self, X, y, pred, cache):
        N = y.shape[0]
        H1, A1, H2, A2, H3 = cache

        dH3 = (pred - y) / N # (N, num_classes)

        dW3 = A2.T @ dH3 # (hidden_layer2, N) @ (N, num_classes) = (hidden_layer2, num_classes)
        db3 = torch.sum(dH3, dim = 0) # (num_classes)
        dA2 = dH3 @ self.W3.T # (N, num_classes) @ (num_classes, hidden_layer2) = (N, hidden_layer2)

        dH2 = self.relu_grad(H2) * dA2 # (N, hidden_layer2)

        dW2 = A1.T @ dH2 # (hidden_layer1, N) @ (N, hidden_layer2) = (hidden_layer1, hidden_layer2)
        db2 = torch.sum(dH2, dim = 0) # (hidden_layer2)
        dA1 = dH2 @ self.W2.T # (N, hidden_layer2) @ (hidden_layer2, hidden_layer1) = (N, hidden_layer1)

        dH1 = self.tanh_grad(A1) # (N, hidden_layer1)

        dW1 = X.T @ dH1 # (input_features, N) @ (N, hidden_layer1) = (input_features, hidden_layer1)
        db1 = torch.sum(dH1, dim = 0) # (hidden_layer1)
        
        self.W1 = self.update_parameters(self.W1, dW1, self.learning_rate)
        self.W2 = self.update_parameters(self.W2, dW2, self.learning_rate)
        self.W3 = self.update_parameters(self.W3, dW3, self.learning_rate)

        self.b1 = self.update_parameters(self.b1, db1, self.learning_rate)
        self.b2 = self.update_parameters(self.b2, db2, self.learning_rate)
        self.b3 = self.update_parameters(self.b3, db3, self.learning_rate)


    def train(self, X_train, y_train, X_val, y_val, iterations = 500):
        for iter in range(iterations):
            train_predictions, train_cache = self.forward(X_train)
            training_loss = self.loss(train_predictions, y_train)

            val_predictions, val_cache = self.forward(X_val)
            validation_loss = self.loss(val_predictions, y_val)
            print(f"Train loss: {round(training_loss, 2)}, val loss: {round(validation_loss, 2)}")
            self.backward(X_train, y_train, train_predictions, train_cache)

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

DATA_DIR = "./data"
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),
])

train_ds = datasets.MNIST(root=DATA_DIR, train=True, download=False, transform=mnist_transform)
test_ds = datasets.MNIST(root=DATA_DIR, train=False, download=False, transform=mnist_transform)
train_loader = DataLoader(train_ds, batch_size=len(train_ds), shuffle=True)
test_loader = DataLoader(test_ds, batch_size=len(test_ds), shuffle=False)

In [7]:
from sklearn.model_selection import train_test_split

X_train_val, y_train_val = next(iter(train_loader))
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2)

X_test, y_test = next(iter(test_loader))
print(f"train shape: {X_train.shape}, val shape: {X_val.shape}, test shape: {X_test.shape}")

train shape: torch.Size([48000, 784]), val shape: torch.Size([12000, 784]), test shape: torch.Size([10000, 784])


In [11]:
NN = NeuralNetwork(X_train. shape[1], 256, 256, 10, -0.1, 1e-2)
NN.train(X_train, y_train, X_val, y_val, iterations = 1000)

RuntimeError: expected m1 and m2 to have the same dtype, but got: float != double